### Gathering Historical Fantasy Scores

In the cells below I will be gathering historical fantasy football players points for a specified season. I will first look at the 2025 season and perform transformations, cleaning, and EDA to determine if more data is needed and identify holes. This data is useful because we can use the historical points as a way to compare out models output performance as in the accuracy of our output.

This data is gathered from ESPN's default-leage endpoint api: [Fantasy API 2025](https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/seasons/2025)

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path


In [4]:
root = Path().cwd().parent
weekly_df = pd.read_csv(root/"data/player_weeks_2025.csv")
weekly_df.head()


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
0,4429795,Jahmyr Gibbs,RB,8,1,15.0,18.419261
1,3117251,Christian McCaffrey,RB,25,1,23.2,17.690153
2,4362628,Ja'Marr Chase,WR,4,1,4.6,19.783577
3,4426515,Puka Nacua,WR,14,1,23.1,16.765436
4,4430807,Bijan Robinson,RB,1,1,24.4,19.748249


In [5]:
weekly_df.shape


(13958, 7)

In [6]:
weekly_df.isna().sum()

player_id              0
player                 0
position               0
pro_team_id            0
week                   0
fantasy_points      2109
projected_points    1202
dtype: int64

In [7]:
weekly_df["player_id"].nunique()

1088

Displaying the Nan values gives us some information as to what is missing. We can see that the only columns with missing data are the fantasy_points column, and the projected_points column. These missing points in both indicate that the palyer didnt play that week and since we are interested in projected points, I will add a column which flags a player if they didnt play in a given week

In [8]:
mask = weekly_df[weekly_df["projected_points"].isna() & weekly_df["fantasy_points"].isna()]

# check why there are no columns with both no point and no projected points
no_proj = weekly_df[weekly_df["projected_points"].isna()]
no_pts = weekly_df[weekly_df["fantasy_points"].isna()]

display(mask.head())
display(no_proj.head())
display(no_pts.head())

print("length of no projected points:", len(no_proj))
print("length of no proj with week = 1:", len(no_proj[no_proj["week"] == 1]))
print(len(no_pts[no_pts["projected_points"] == 0]))

,player_id,player,position,pro_team_id,week,fantasy_points,projected_points


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
355,4601080,Raheim Sanders,RB,5,1,6.3,NaN
422,4430656,Myles Price,WR,16,1,0.0,NaN
450,4715355,Darius Cooper,WR,21,1,0.0,NaN
453,5160110,Jahdae Walker,WR,3,1,0.0,NaN
742,4876006,Zaire Mitchell-Paden,TE,18,1,0.0,NaN


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
144,4240657,Michael Carter,RB,22,1,NaN,0.0
149,4574716,Harrison Mevis,K,14,1,NaN,0.0
150,4430968,Kimani Vidal,RB,24,1,NaN,0.0
167,4034949,Eddy Pineiro,K,25,1,NaN,0.0
227,4569682,Audric Estime,RB,18,1,NaN,0.0


length of no projected points: 1202
length of no proj with week = 1: 9
2101


We can see that there are actually no rows where there are not fantasy points and no projected points, so i displayed both tables to see what is going on. We can see that if there are no fantasy points, then the projected points are 0 instead of nan. And we see there are some cases where there are fantasy points, but no projections. For the case where the fantasy points are NaN and the projected points are 0, I will remove these rows since they are of no use. For the rows which there are fantasy points and no projection, I will impute these values.